In [68]:
pip install numpy pandas matplotlib seaborn scipy scikit-learn imbalanced-learn xgboost lightgbm

Note: you may need to restart the kernel to use updated packages.


In [69]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
)
from scipy.stats import ks_2samp
from imblearn.over_sampling import SMOTE

print("✓ Imports OK")

✓ Imports OK


In [70]:
sample_sub = pd.read_csv('sample_submission.csv')

In [71]:
test_identity = pd.read_csv('test_identity.csv')

In [72]:
test_transaction = pd.read_csv('test_transaction.csv')

In [73]:
train_identity = pd.read_csv('train_identity.csv')

In [74]:
train_transaction = pd.read_csv('train_transaction.csv')

In [75]:
sample_sub

,TransactionID,isFraud
0,3663549,0.5
1,3663550,0.5
2,3663551,0.5
3,3663552,0.5
4,3663553,0.5
...,...,...
506686,4170235,0.5
506687,4170236,0.5
506688,4170237,0.5
506689,4170238,0.5


In [76]:
test_identity

,TransactionID,id-01,id-02,id-03,id-04,id-05,id-06,id-07,id-08,id-09,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663586,-45.0,280290.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13 Build/HUAWEIMYA-L13
1,3663588,0.0,3579.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 67.0 for android,24.0,1280x720,match_status:2,T,F,T,T,mobile,LGLS676 Build/MXB48T
2,3663597,-5.0,185210.0,NaN,NaN,1.0,0.0,NaN,NaN,NaN,...,ie 11.0 for tablet,NaN,NaN,NaN,F,T,T,F,desktop,Trident/7.0
3,3663601,-45.0,252944.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13 Build/HUAWEIMYA-L13
4,3663602,-95.0,328680.0,NaN,NaN,7.0,-33.0,NaN,NaN,NaN,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-G9650 Build/R16NW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141902,4170230,-20.0,473365.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,chrome 71.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-J700M
141903,4170233,-5.0,489917.0,0.0,0.0,-4.0,-32.0,NaN,NaN,0.0,...,chrome 71.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-J320M
141904,4170234,-5.0,110081.0,NaN,NaN,22.0,-31.0,NaN,NaN,NaN,...,mobile safari 10.0,32.0,1334x750,match_status:2,T,F,F,T,mobile,iOS Device
141905,4170236,-45.0,266704.0,NaN,NaN,-3.0,-10.0,NaN,NaN,NaN,...,chrome 43.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,ALE-L23 Build/HuaweiALE-L23


In [77]:
train_identity

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144228,3577521,-15.0,145955.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 66.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,F3111 Build/33.3.A.1.97
144229,3577526,-5.0,172059.0,NaN,NaN,1.0,-5.0,NaN,NaN,NaN,...,chrome 55.0 for android,32.0,855x480,match_status:2,T,F,T,F,mobile,A574BL Build/NMF26F
144230,3577529,-20.0,632381.0,NaN,NaN,-1.0,-36.0,NaN,NaN,NaN,...,chrome 65.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,Moto E (4) Plus Build/NMA26.42-152
144231,3577531,-5.0,55528.0,0.0,0.0,0.0,-7.0,NaN,NaN,0.0,...,chrome 66.0,24.0,2560x1600,match_status:2,T,F,T,F,desktop,MacOS


In [78]:
train_transaction

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
# ─── MERGE: transaction + identity ───────────────────────────────────────────
# Left join: nem toda transação tem dados de identidade (~144k de 590k)
train = train_transaction.merge(train_identity, on='TransactionID', how='left')

print(f"Shape após merge: {train.shape}")
print(f"Transações COM identidade: {train['DeviceType'].notna().sum():,} ({train['DeviceType'].notna().mean():.1%})")
print(f"Transações SEM identidade: {train['DeviceType'].isna().sum():,}")

Shape após merge: (590540, 434)
Transações COM identidade: 140,810 (23.8%)
Transações SEM identidade: 449,730


In [80]:
# ─── EDA RÁPIDA ──────────────────────────────────────────────────────────────
# 1. Taxa de fraude base (apenas para referência — não usamos como target)
print(f"Taxa de fraude: {train['isFraud'].mean():.2%}\n")

# 2. Distribuição de valores de transação
print("TransactionAmt:")
print(train['TransactionAmt'].describe())
print()

# 3. Colunas com mais de 50% nulo (candidatas a exclusão)
missing = (train.isnull().mean() * 100).sort_values(ascending=False)
high_missing = missing[missing > 50]
print(f"Colunas com >50% nulo: {len(high_missing)}")
print(high_missing.head(10))

Taxa de fraude: 3.50%

TransactionAmt:
count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64

Colunas com >50% nulo: 214
id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
dtype: float64


In [81]:
# ─── SEPARAR GRUPOS DE FEATURES ──────────────────────────────────────────────
# Entender o que temos antes de criar novas variáveis

# Features de negócio (raw, interpretáveis)
cols_negocio = ['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3',
                'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2',
                'P_emaildomain', 'R_emaildomain']

# Features de identidade
cols_identidade = [c for c in train.columns if c.startswith('id')]

# Features V (anonimizadas — usaremos só as com baixo missingness)
cols_v = [c for c in train.columns if c.startswith('V')]
cols_v_ok = [c for c in cols_v if train[c].isnull().mean() < 0.5]

print(f"Features de negócio: {len(cols_negocio)}")
print(f"Features de identidade: {len(cols_identidade)}")
print(f"Features V totais: {len(cols_v)} | com <50% nulo: {len(cols_v_ok)}")

Features de negócio: 14
Features de identidade: 38
Features V totais: 339 | com <50% nulo: 180


In [82]:
# ─── BLOCO 1: Variáveis de tempo ─────────────────────────────────────────────
# TransactionDT é tempo em segundos desde um ponto de referência (não é unix timestamp)
# Extraímos hora e dia da semana para capturar padrão comportamental temporal

train['hora'] = (train['TransactionDT'] // 3600) % 24
train['dia_semana'] = (train['TransactionDT'] // (3600 * 24)) % 7
train['is_madrugada'] = ((train['hora'] >= 0) & (train['hora'] <= 5)).astype(int)

In [83]:
# ─── BLOCO 2: Fingerprint do portador — card1 como chave de agregação ─────────
# card1 é o identificador mais próximo de "cliente" nesse dataset
# Agrupamos o histórico de cada card1 para criar variáveis de comportamento passado

agg_card1 = train.groupby('card1').agg(
    card1_qtd_transacoes    = ('TransactionAmt', 'count'),
    card1_valor_medio       = ('TransactionAmt', 'mean'),
    card1_valor_total       = ('TransactionAmt', 'sum'),
    card1_valor_max         = ('TransactionAmt', 'max'),
    card1_valor_std         = ('TransactionAmt', 'std'),
    card1_produtos_unicos   = ('ProductCD', 'nunique'),
    card1_enderecos_unicos  = ('addr1', 'nunique'),
).reset_index()

train = train.merge(agg_card1, on='card1', how='left')

In [84]:
# ─── BLOCO 3: Desvio da transação em relação ao histórico do cartão ───────────
# A transação atual comparada ao comportamento médio daquele cartão

train['valor_vs_media_card1'] = train['TransactionAmt'] / (train['card1_valor_medio'] + 1)
train['valor_vs_max_card1']   = train['TransactionAmt'] / (train['card1_valor_max'] + 1)
train['is_valor_acima_2std']  = (
    train['TransactionAmt'] > (train['card1_valor_medio'] + 2 * train['card1_valor_std'].fillna(0))
).astype(int)

In [85]:
# ─── BLOCO 4: Concentração por produto (equivalente à concentração por estabelecimento) ──
# ProductCD é a categoria do produto — W, H, C, S, R
# Concentração alta = cliente comprando só uma coisa, possível compra forçada

concentracao_produto = train.groupby('card1')['ProductCD'].apply(
    lambda x: x.value_counts(normalize=True).max()
).reset_index()
concentracao_produto.columns = ['card1', 'concentracao_produto']

train = train.merge(concentracao_produto, on='card1', how='left')

In [86]:
# ─── BLOCO 5: Features de identidade — diversidade de dispositivos ─────────────
# Quantos dispositivos diferentes esse cartão já usou?

agg_device = train.groupby('card1').agg(
    card1_devices_unicos    = ('DeviceType', 'nunique'),
    card1_browsers_unicos   = ('id_31', 'nunique'),
).reset_index()

train = train.merge(agg_device, on='card1', how='left')
train['card1_devices_unicos']  = train['card1_devices_unicos'].fillna(1)
train['card1_browsers_unicos'] = train['card1_browsers_unicos'].fillna(1)

In [89]:
train.to_csv('analitico.csv')

In [87]:
# ─── BLOCO 6: Definir feature set final para o K-Means ───────────────────────
features_kmeans = [
    'valor_vs_media_card1',
    'valor_vs_max_card1',
    'is_valor_acima_2std',
    'card1_valor_std',
    'card1_qtd_transacoes',
    'card1_enderecos_unicos',
    'card1_devices_unicos',
    'concentracao_produto',
    'is_madrugada',
]

df_model = train[features_kmeans].fillna(0)
print(f"Shape do dataset para modelagem: {df_model.shape}")
print(df_model.describe().round(2))

Shape do dataset para modelagem: (590540, 9)
       valor_vs_media_card1  valor_vs_max_card1  is_valor_acima_2std  \
count             590540.00           590540.00            590540.00   
mean                   0.99                0.13                 0.04   
std                    1.36                0.21                 0.19   
min                    0.00                0.00                 0.00   
25%                    0.38                0.02                 0.00   
50%                    0.63                0.05                 0.00   
75%                    1.09                0.13                 0.00   
max                  113.72                1.00                 1.00   

       card1_valor_std  card1_qtd_transacoes  card1_enderecos_unicos  \
count        590540.00             590540.00               590540.00   
mean            185.63               2528.82                   26.13   
std             131.07               3702.66                   19.67   
min               

In [88]:
df_model.to_csv('base_a_classificar.csv')